## Import all required libraries

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Prepare time series data

In [ ]:
# Simulate monthly house price index (use real data in practice)
np.random.seed(42)
n_months = 120  # 10 years of monthly data
trend     = np.linspace(200000, 400000, n_months)
seasonal  = 15000 * np.sin(np.linspace(0, 8*np.pi, n_months))
noise     = np.random.randn(n_months) * 5000
prices    = trend + seasonal + noise

# Scale to 0-1 — REQUIRED for LSTM
scaler = MinMaxScaler()
prices_scaled = scaler.fit_transform(prices.reshape(-1,1)).flatten()

# Create sequences: use past 12 months to predict next month
LOOKBACK = 12

def create_sequences(data, lookback):
    X, y = [], []
    for i in range(len(data) - lookback):
        X.append(data[i:i+lookback])   # 12 months of history
        y.append(data[i+lookback])     # next month's price
    return np.array(X), np.array(y)

X_seq, y_seq = create_sequences(prices_scaled, LOOKBACK)

# Train/test split — NO shuffling for time series
split = int(0.8 * len(X_seq))
X_train = torch.FloatTensor(X_seq[:split]).unsqueeze(-1)  # (samples, 12, 1)
y_train = torch.FloatTensor(y_seq[:split])
X_test  = torch.FloatTensor(X_seq[split:]).unsqueeze(-1)
y_test  = torch.FloatTensor(y_seq[split:])

print(f"X_train shape: {X_train.shape}")  # (96, 12, 1)
print(f"y_train shape: {y_train.shape}")  # (96,)

## LSTM Model

In [ ]:
class HousePriceLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=64,
                 num_layers=2, dropout=0.2):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size  = input_size,   # 1 feature (price)
            hidden_size = hidden_size,  # 64 memory cells
            num_layers  = num_layers,   # 2 stacked LSTM layers
            dropout     = dropout,      # dropout between layers
            batch_first = True          # (batch, seq, feature) format
        )
        self.fc = nn.Linear(hidden_size, 1)  # LSTM output → price

    def forward(self, x):
        # x shape: (batch, seq_len, input_size)
        lstm_out, (h_n, c_n) = self.lstm(x)
        # lstm_out: (batch, seq_len, hidden_size)
        # We only want the LAST time step's output
        last_output = lstm_out[:, -1, :]     # (batch, hidden_size)
        prediction  = self.fc(last_output)   # (batch, 1)
        return prediction.squeeze(-1)

lstm_model = HousePriceLSTM().to(device)
print(f"LSTM Parameters: {sum(p.numel() for p in lstm_model.parameters()):,}")

## Train LSTM

In [ ]:
import torch.optim as optim
optimizer = optim.Adam(lstm_model.parameters(), lr=0.001)
criterion = nn.MSELoss()

X_train_d = X_train.to(device)
y_train_d = y_train.to(device)
X_test_d  = X_test.to(device)

train_losses, test_losses = [], []

for epoch in range(100):
    lstm_model.train()
    optimizer.zero_grad()
    pred  = lstm_model(X_train_d)
    loss  = criterion(pred, y_train_d)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(lstm_model.parameters(), 1.0)
    optimizer.step()
    train_losses.append(loss.item())

    lstm_model.eval()
    with torch.no_grad():
        test_pred = lstm_model(X_test_d)
        test_loss = criterion(test_pred, y_test.to(device))
    test_losses.append(test_loss.item())

    if (epoch+1) % 20 == 0:
        print(f"Epoch {epoch+1}: Train {loss.item():.5f} | Test {test_loss.item():.5f}")

# Inverse transform and visualize
lstm_model.eval()
with torch.no_grad():
    predictions_scaled = lstm_model(X_test_d).cpu().numpy()

predictions = scaler.inverse_transform(predictions_scaled.reshape(-1,1))
actuals     = scaler.inverse_transform(y_test.numpy().reshape(-1,1))

plt.figure(figsize=(12, 5))
plt.plot(actuals,     label='Actual Price',    color='#1D9E75')
plt.plot(predictions, label='LSTM Prediction', color='#D85A30')
plt.title('LSTM House Price Forecasting')
plt.xlabel('Month'); plt.ylabel('Price ($)')
plt.legend(); plt.show()

Critical Time Series Rule

NEVER shuffle time series data

**Wrong:**

train_test_split(X, y, shuffle=True)   # ❌ leaks future into past

**Right:**

split = int(0.8 * len(X))

X_train, X_test = X[:split], X[split:]  # ✅ time
order preserved